In [ ]:
import numpy as np
import pandas as pd
from lenskit import batch, topn, util
import os


In [ ]:
PROCESSED_FILE_DIR = "/home/shoaib/recommender-system/royal/journal-revision/New/Politifact/processed-files"
DATA_FILE_DIR = "/home/shoaib/recommender-system/royal/journal-revision/New/Politifact/data-files"
RESULTS_DIR = "/home/shoaib/recommender-system/royal/journal-revision/New/Politifact/results"

In [ ]:
train=pd.read_csv(f"{DATA_FILE_DIR}/train_ratings.csv").iloc[:,[1,2,3]]

In [3]:
train.drop_duplicates(subset=['user', 'item'], inplace=True)

In [4]:
train


,user,item,ratings
0,1,288,1.0
1,1,353,2.0
2,1,435,1.5
3,1,184,1.5
4,1,416,1.5
...,...,...,...
19236,1028,242,0.5
19237,0,47,1.5
19238,0,51,1.5
19239,0,231,1.5


In [5]:
user_item_sets = train.groupby('user')['item'].apply(set).to_dict()

In [6]:
user_item_sets

{0: {47, 51, 231, 484},
 1: {3, 135, 184, 275, 288, 353, 395, 416, 435, 439, 458},
 2: {25, 26, 251, 319, 347, 352, 440, 456, 519, 522},
 3: {25,
  29,
  32,
  124,
  139,
  189,
  263,
  274,
  348,
  352,
  380,
  410,
  434,
  440,
  456,
  489,
  500,
  505},
 4: {9,
  25,
  29,
  67,
  127,
  130,
  187,
  189,
  190,
  227,
  355,
  359,
  389,
  434,
  511,
  531},
 5: {25,
  31,
  71,
  135,
  171,
  213,
  221,
  240,
  275,
  324,
  334,
  345,
  359,
  387,
  395,
  476},
 6: {25, 92, 224, 259, 260, 261, 324, 387, 394, 421, 456},
 7: {9, 25, 92, 223, 224, 288, 359, 394},
 8: {25, 31, 304, 356, 360, 387, 476, 524, 530},
 9: {21, 81, 148, 195, 250, 308, 346, 408},
 10: {20,
  22,
  86,
  106,
  109,
  113,
  127,
  171,
  173,
  175,
  187,
  204,
  206,
  223,
  269,
  282,
  300,
  323,
  334,
  360,
  382,
  412,
  432,
  450},
 11: {31, 53, 148, 195, 205, 221, 250, 263, 275, 395, 530},
 12: {21, 95, 173, 242, 269, 321, 376, 476},
 13: {44, 84, 111, 158, 221, 230, 239, 293,

In [7]:
def calculate_cosine_similarity(user1, user2, user_item_sets):
    items_user1 = user_item_sets.get(user1, set())
    items_user2 = user_item_sets.get(user2, set())
    
    # Calculate intersection of items
    intersection = len(items_user1.intersection(items_user2))
    
    # Compute the dot product of the vectors representing the item sets
    dot_product = intersection
    
    # Compute the magnitudes of the vectors
    magnitude_user1 = len(items_user1)
    magnitude_user2 = len(items_user2)
    
    # Handle division by zero
    if magnitude_user1 == 0 or magnitude_user2 == 0:
        return 0.0
    
    # Compute cosine similarity
    cosine_similarity = dot_product / (magnitude_user1 * magnitude_user2) ** 0.5
    return cosine_similarity



In [8]:
unique_users = train['user'].unique()

In [9]:
# number of elements
print(len(unique_users))

# equivalent for numpy arrays
print(unique_users.size)

# shape (useful for multi-dim arrays)
print(unique_users.shape)

1029
1029
(1029,)


In [10]:
from itertools import product


In [11]:
# Calculate Jaccard similarity for all user pairs
cosine_similarities = []

for user1, user2 in product(unique_users, repeat=2):
    if user1 != user2:
        similarity = calculate_cosine_similarity(user1, user2,user_item_sets)
        cosine_similarities.append({'User1': user1, 'User2': user2, 'Similarity': similarity})

In [12]:
# Create a DataFrame for Jaccard similarities
cosine_df = pd.DataFrame(cosine_similarities)

# Reshape the DataFrame to have User1, User2, and Similarity as columns
cosine_matrix = cosine_df.pivot(index='User1', columns='User2', values='Similarity')

idx = range(len(cosine_matrix))
cosine_matrix.values[idx, idx] = 1.0


In [13]:
cosine_df 

,User1,User2,Similarity
0,1,2,0.000000
1,1,3,0.000000
2,1,4,0.000000
3,1,5,0.226134
4,1,6,0.000000
...,...,...,...
1057807,0,1024,0.000000
1057808,0,1025,0.000000
1057809,0,1026,0.000000
1057810,0,1027,0.000000


In [14]:
cosine_matrix

User2,0,1,2,3,4,5,6,7,8,9,...,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028
User1,,,,,,,,,,,,,,,,,,,,,
0,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.0,1.000000,0.000000,0.000000,0.000000,0.226134,0.000000,0.106600,0.000000,0.000000,...,0.261116,0.090909,0.000000,0.000000,0.000000,0.000000,0.000000,0.128565,0.064282,0.000000
2,0.0,0.000000,1.000000,0.298142,0.079057,0.079057,0.190693,0.111803,0.105409,0.000000,...,0.091287,0.095346,0.000000,0.000000,0.119523,0.000000,0.000000,0.269680,0.000000,0.084515
3,0.0,0.000000,0.298142,1.000000,0.235702,0.058926,0.142134,0.083333,0.078567,0.000000,...,0.272166,0.213201,0.000000,0.000000,0.178174,0.223607,0.000000,0.201008,0.050252,0.000000
4,0.0,0.000000,0.079057,0.235702,1.000000,0.125000,0.075378,0.265165,0.083333,0.000000,...,0.000000,0.000000,0.106600,0.000000,0.094491,0.158114,0.149071,0.053300,0.053300,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1024,0.0,0.000000,0.000000,0.223607,0.158114,0.000000,0.095346,0.000000,0.000000,0.000000,...,0.182574,0.000000,0.067420,0.000000,0.119523,1.000000,0.000000,0.134840,0.067420,0.000000
1025,0.0,0.000000,0.000000,0.000000,0.149071,0.186339,0.089893,0.210819,0.049690,0.105409,...,0.000000,0.000000,0.286039,0.188562,0.056344,0.000000,1.000000,0.031782,0.095346,0.119523
1026,0.0,0.128565,0.269680,0.201008,0.053300,0.000000,0.128565,0.150756,0.000000,0.000000,...,0.184637,0.192847,0.045455,0.000000,0.080582,0.134840,0.031782,1.000000,0.045455,0.170941


In [ ]:
key = pd.read_csv(f"{DATA_FILE_DIR}/politifact_Shu_fake_news_keyforSOCIALMF.csv")

In [17]:
key_news_id=key[["news_id_new","news_label"]].drop_duplicates(subset=["news_id_new"],keep="last").reset_index(drop=True)

In [18]:
key_news_id["fake_real_labels"]=key_news_id["news_label"].apply(lambda x:0 if x=="fake" else 1)

In [19]:
train= pd.merge(train,key_news_id,left_on="item",right_on="news_id_new", how='inner')

In [20]:
train=train.iloc[:,[0,1,2,4,5]]

In [21]:
interaction_counts = train.groupby(['user', 'news_label']).size().unstack(fill_value=0)
interaction_counts.rename(columns={'fake': 'fake_news_count', 'real': 'real_news_count'}, inplace=True)
interaction_counts=interaction_counts.reset_index()

In [22]:
interaction_counts=interaction_counts.loc[:,['user','fake_news_count','real_news_count']]

In [23]:
def reputation_user(alpha, beta):
    """
    Calculate the ratio (alpha + 1) / (alpha + beta + 2).

    Parameters:
    alpha (float): The alpha parameter.
    beta (float): The beta parameter.

    Returns:
    float: The calculated ratio.
    """
    ratio = (alpha) / (alpha + beta)
    return ratio

In [24]:
interaction_counts['reputation'] = (interaction_counts['real_news_count'])/ (interaction_counts['real_news_count'] + interaction_counts['fake_news_count'])


In [25]:
interaction_counts

news_label,user,fake_news_count,real_news_count,reputation
0,0,2,2,0.500000
1,1,3,8,0.727273
2,2,2,8,0.800000
3,3,4,14,0.777778
4,4,12,4,0.250000
...,...,...,...,...
1024,1024,1,9,0.900000
1025,1025,45,0,0.000000
1026,1026,5,17,0.772727
1027,1027,17,5,0.227273


In [26]:
from collections import defaultdict

In [27]:
# Filter users based on reputation greater than 0.5
reputation_threshold = 0.6

trusted_users = interaction_counts[interaction_counts['reputation'] > reputation_threshold]['user']

untrusted_users = interaction_counts[interaction_counts['reputation'] <= reputation_threshold]['user'] # Save as a untrusted user list and use it to the social space ss60

# Dictionary to store top similar trustworthy users for each user
top_trustworthy_similar_users = defaultdict(set)

# Iterate over each user
for user in cosine_matrix.index:
    # Get cosine similarity values for the user
    user_similarity = cosine_matrix.loc[user]
    # Exclude untrustworthy similar users (reputation <= 0.6) and the user itself
    similar_trustworthy_users = user_similarity[(user_similarity > 0) & (interaction_counts.loc[user_similarity.index, 'reputation'] > reputation_threshold) & (user_similarity.index != user)]
    # Sort similar trustworthy users by similarity (descending order) and take the top 10
    temp=set(similar_trustworthy_users.sort_values(ascending=False).index[:10].tolist())
    if len(temp)>0:
        top_trustworthy_similar_users[user] =set(similar_trustworthy_users.sort_values(ascending=False).index[:10].tolist()) 

In [28]:
# Convert to a simple Python list
untrusted_users_list_ss60 = untrusted_users.tolist()

# Save as pickle file
import pickle
with open('/home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/processed-files/untrusted_users_ss60.pickle', 'wb') as f:
    pickle.dump(untrusted_users_list_ss60, f)

print(f"Saved {len(untrusted_users_list_ss60)} untrusted users")
print("Preview:", untrusted_users_list_ss60[:10])  # Show first 10

# Later, to load it:
# with open('untrusted_users.pickle', 'rb') as f:
#     untrusted_users_list = pickle.load(f)

Saved 660 untrusted users
Preview: [0, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [29]:
top_trustworthy_similar_users

defaultdict(set,
            {1: {96, 168, 239, 266, 267, 289, 591, 717, 744, 913},
             2: {124, 224, 289, 429, 432, 452, 528, 682, 698, 916},
             3: {26, 161, 272, 307, 349, 362, 463, 521, 762, 956},
             4: {3, 112, 236, 279, 362, 431, 463, 653, 956, 1015},
             5: {1, 130, 267, 425, 502, 591, 614, 717, 744, 840},
             6: {72, 147, 333, 362, 397, 537, 791, 956, 962, 985},
             7: {164, 266, 279, 289, 333, 362, 471, 573, 591, 791},
             8: {44, 130, 165, 357, 454, 698, 713, 778, 956, 985},
             9: {60, 113, 266, 388, 425, 527, 563, 570, 865, 909},
             10: {97, 185, 583, 591, 649, 796, 849, 865, 991, 1024},
             11: {1, 130, 258, 425, 554, 614, 717, 887, 909, 919},
             12: {326, 334, 411, 540, 879, 1010},
             13: {44, 60, 81, 160, 440, 442, 515, 698, 703, 864},
             14: {189, 229, 288, 440, 596, 600, 637, 849, 879, 909},
             15: {78, 189, 217, 229, 440, 471, 596, 837, 8

In [30]:
import pickle

# Specify the path to your pickle file
pickle_file_path = "/home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/processed-files/TrustWorthy_Politifact_MAGrec.pickle"

# Open the pickle file in read binary mode
with open(pickle_file_path, 'rb') as file:
    # Load the data from the pickle file
    loaded_data = pickle.load(file)

In [31]:
history_u_lists, history_ur_lists, history_v_lists, history_vr_lists, train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list=loaded_data

In [32]:
social_adj_lists=top_trustworthy_similar_users

In [33]:
social_adj_lists

defaultdict(set,
            {1: {96, 168, 239, 266, 267, 289, 591, 717, 744, 913},
             2: {124, 224, 289, 429, 432, 452, 528, 682, 698, 916},
             3: {26, 161, 272, 307, 349, 362, 463, 521, 762, 956},
             4: {3, 112, 236, 279, 362, 431, 463, 653, 956, 1015},
             5: {1, 130, 267, 425, 502, 591, 614, 717, 744, 840},
             6: {72, 147, 333, 362, 397, 537, 791, 956, 962, 985},
             7: {164, 266, 279, 289, 333, 362, 471, 573, 591, 791},
             8: {44, 130, 165, 357, 454, 698, 713, 778, 956, 985},
             9: {60, 113, 266, 388, 425, 527, 563, 570, 865, 909},
             10: {97, 185, 583, 591, 649, 796, 849, 865, 991, 1024},
             11: {1, 130, 258, 425, 554, 614, 717, 887, 909, 919},
             12: {326, 334, 411, 540, 879, 1010},
             13: {44, 60, 81, 160, 440, 442, 515, 698, 703, 864},
             14: {189, 229, 288, 440, 596, 600, 637, 849, 879, 909},
             15: {78, 189, 217, 229, 440, 471, 596, 837, 8

In [34]:
import pickle
# Organize the variables into a tuple in the desired order
data_to_save = (history_u_lists, history_ur_lists, history_v_lists, history_vr_lists,
                train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list)

# Specify the path to save the pickle file
output_pickle_file_path = "/home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/processed-files/D1_Trust_Neighbor_Update_0.6_threshold.pickle"

# Open the pickle file in write binary mode ('wb')
with open(output_pickle_file_path, 'wb') as file:
    # Dump the data to the pickle file
    pickle.dump(data_to_save, file)

print("Data saved to pickle file:", output_pickle_file_path)


Data saved to pickle file: /home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/processed-files/D1_Trust_Neighbor_Update_0.6_threshold.pickle


## Training 

In [37]:
import torch
import torch.nn as nn
import pickle
import numpy as np
import argparse
import os
from UV_Encoders import UV_Encoder
from UV_Aggregators import UV_Aggregator
from Social_Encoders import Social_Encoder
from Social_Aggregators import Social_Aggregator


class GraphRec(nn.Module):
    def __init__(self, enc_u, enc_v_history, r2e):
        super(GraphRec, self).__init__()
        self.enc_u = enc_u
        self.enc_v_history = enc_v_history
        self.embed_dim = enc_u.embed_dim
        self.w_ur1 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_ur2 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_vr1 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_vr2 = nn.Linear(self.embed_dim, self.embed_dim)
        self.w_uv1 = nn.Linear(self.embed_dim * 2, self.embed_dim)
        self.w_uv2 = nn.Linear(self.embed_dim, 16)
        self.w_uv3 = nn.Linear(16, 1)
        self.r2e = r2e
        self.bn1 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn2 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn3 = nn.BatchNorm1d(self.embed_dim, momentum=0.5)
        self.bn4 = nn.BatchNorm1d(16, momentum=0.5)

    def forward(self, nodes_u, nodes_v):
        embeds_u = self.enc_u(nodes_u)
        embeds_v = self.enc_v_history(nodes_v)
        scores_u = torch.matmul(embeds_u, embeds_v.t())
        return scores_u.squeeze()

    def bpr_loss(self, nodes_u, nodes_v, device, num_items):
        scores_u = self.forward(nodes_u, nodes_v)
        nodes_v_negative = torch.randint(0, num_items, size=nodes_u.size(), dtype=torch.long).to(device)
        scores_u_negative = self.forward(nodes_u, nodes_v_negative)
        bpr_loss = -torch.log(torch.sigmoid(scores_u - scores_u_negative)).mean()
        return bpr_loss

def train(model, device, train_loader, optimizer, epoch, num_items):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        batch_nodes_u, batch_nodes_v, _ = data
        optimizer.zero_grad()
        loss = model.bpr_loss(batch_nodes_u.to(device), batch_nodes_v.to(device), device, num_items)
        loss.backward(retain_graph=True)
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 0:
            print('[%d, %5d] loss: %.3f' % (epoch, i, running_loss / 100))
            running_loss = 0.0

def get_top_100_recommendations(model, device, test_loader, num_items, k=100):
    model.eval()
    user_recommendations_dict = {}  # Dictionary to store user recommendations

    with torch.no_grad():
        for test_u, test_v, _ in test_loader:
            test_u, test_v = test_u.to(device), test_v.to(device)
            scores = model.forward(test_u, test_v)
            scores = scores.squeeze().tolist()
            top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

            user_ids = test_u.tolist()  # Convert tensor to list

            for user_id in user_ids:
                user_recommendations = [(rank, test_v[item_index].item()) for rank, item_index in enumerate(top_indices, start=1)]
                user_recommendations_dict[user_id] = user_recommendations

    return user_recommendations_dict




def main():
    parser = argparse.ArgumentParser(description='Social Recommendation: GraphRec model')
    parser.add_argument('--batch_size', type=int, default=128, metavar='N', help='input batch size for training')
    parser.add_argument('--embed_dim', type=int, default=64, metavar='N', help='embedding size')
    parser.add_argument('--lr', type=float, default=0.001, metavar='LR', help='learning rate')
    parser.add_argument('--test_batch_size', type=int, default=1000, metavar='N', help='input batch size for testing')
    parser.add_argument('--epochs', type=int, default=100, metavar='N', help='number of epochs to train')
    args, unknown = parser.parse_known_args()

    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    use_cuda = False
    if torch.cuda.is_available():
        use_cuda = True
    device = torch.device("cuda" if use_cuda else "cpu")

    embed_dim = args.embed_dim
    dir_data = '/home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/processed-files/D1_Trust_Neighbor_Update_0.6_threshold'

    path_data = dir_data + ".pickle"
    data_file = open(path_data, 'rb')
    history_u_lists, history_ur_lists, history_v_lists, history_vr_lists, train_u, train_v, train_r, test_u, test_v, test_r, social_adj_lists, ratings_list = pickle.load(
        data_file)
    total_num_items = len(set(train_v + test_v))

    trainset = torch.utils.data.TensorDataset(torch.LongTensor(train_u), torch.LongTensor(train_v),
                                              torch.FloatTensor(train_r))
    testset = torch.utils.data.TensorDataset(torch.LongTensor(test_u), torch.LongTensor(test_v),
                                             torch.FloatTensor(test_r))
    train_loader = torch.utils.data.DataLoader(trainset, batch_size=args.batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(testset, batch_size=args.test_batch_size, shuffle=True)
    num_users = history_u_lists.__len__()
    num_items = history_v_lists.__len__()

    u2e = nn.Embedding(num_users, embed_dim).to(device)
    v2e = nn.Embedding(num_items, embed_dim).to(device)
    r2e = nn.Embedding(total_num_items, embed_dim).to(device)  # Define r2e with the correct number of ratings
    total_num_items = len(set(train_v + test_v))

    agg_u_history = UV_Aggregator(v2e, r2e, u2e, embed_dim, cuda=device, uv=True)
    enc_u_history = UV_Encoder(u2e, embed_dim, history_u_lists, history_ur_lists, agg_u_history, cuda=device, uv=True)
    agg_u_social = Social_Aggregator(lambda nodes: enc_u_history(nodes).t(), u2e, embed_dim, cuda=device)
    enc_u = Social_Encoder(lambda nodes: enc_u_history(nodes).t(), embed_dim, social_adj_lists, agg_u_social,
                           base_model=enc_u_history, cuda=device)

    agg_v_history = UV_Aggregator(v2e, r2e, u2e, embed_dim, cuda=device, uv=False)
    enc_v_history = UV_Encoder(v2e, embed_dim, history_v_lists, history_vr_lists, agg_v_history, cuda=device, uv=False)

    graphrec = GraphRec(enc_u, enc_v_history, r2e).to(device)
    optimizer = torch.optim.RMSprop(graphrec.parameters(), lr=args.lr, alpha=0.9)

    for epoch in range(1, args.epochs + 1):
        train(graphrec, device, train_loader, optimizer, epoch, total_num_items)

    top_100_recommendations = get_top_100_recommendations(graphrec, device, test_loader, num_items=total_num_items)

    # Print or use top_100_recommendations as needed
    return top_100_recommendations

if __name__ == "__main__":
    recommendation_list=main()


[1,     0] loss: 0.007
[1,   100] loss: 0.478
[2,     0] loss: 0.004
[2,   100] loss: 0.421
[3,     0] loss: 0.005
[3,   100] loss: 0.411
[4,     0] loss: 0.005
[4,   100] loss: 0.406
[5,     0] loss: 0.004
[5,   100] loss: 0.406
[6,     0] loss: 0.004
[6,   100] loss: 0.397
[7,     0] loss: 0.004
[7,   100] loss: 0.393
[8,     0] loss: 0.005
[8,   100] loss: 0.400
[9,     0] loss: 0.004
[9,   100] loss: 0.398
[10,     0] loss: 0.004
[10,   100] loss: 0.397
[11,     0] loss: 0.004
[11,   100] loss: 0.401
[12,     0] loss: 0.004
[12,   100] loss: 0.397
[13,     0] loss: 0.004
[13,   100] loss: 0.399
[14,     0] loss: 0.004
[14,   100] loss: 0.399
[15,     0] loss: 0.004
[15,   100] loss: 0.392
[16,     0] loss: 0.005
[16,   100] loss: 0.399
[17,     0] loss: 0.004
[17,   100] loss: 0.394
[18,     0] loss: 0.004
[18,   100] loss: 0.395
[19,     0] loss: 0.004
[19,   100] loss: 0.397
[20,     0] loss: 0.005
[20,   100] loss: 0.394
[21,     0] loss: 0.004
[21,   100] loss: 0.392
[22,     0

In [38]:
rows = []
for user, rankings in recommendation_list.items():
    for rank, item in rankings:
        rows.append({"user": user, "rank": rank, "item_update": item, "item": item+1})

all_recs = pd.DataFrame(rows)

In [39]:
cold_start_item=[47, 51, 231, 484]

In [40]:
filtered_recs = all_recs[~all_recs['item'].isin(cold_start_item)]
filtered_recs

,user,rank,item_update,item
0,341,1,409,410
1,341,2,455,456
2,341,3,531,532
3,341,4,359,360
4,341,5,262,263
...,...,...,...,...
100836,1021,25,318,319
100837,1021,26,116,117
100838,1021,27,69,70
100839,1021,28,28,29


In [41]:
filtered_recs['rank'] = filtered_recs['rank'].astype(int)

/tmp/ipykernel_1390737/435666994.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_recs['rank'] = filtered_recs['rank'].astype(int)


In [42]:
filtered_recs['re-rank'] = filtered_recs.groupby('user')['rank'].rank(method='first')
filtered_recs['rank'] = filtered_recs['re-rank'].astype(int)
filtered_recs=filtered_recs.iloc[:,[0,1,2,3]]

/tmp/ipykernel_1390737/347894999.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_recs['re-rank'] = filtered_recs.groupby('user')['rank'].rank(method='first')
/tmp/ipykernel_1390737/347894999.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_recs['rank'] = filtered_recs['re-rank'].astype(int)


In [ ]:
# Save filtered_recs to pickle
filtered_recs.to_pickle(f'{PROCESSED_FILE_DIR}/Politifact_filtered_recs_trust_neighbour_0.6_threshold.pickle')

In [43]:
filtered_recs

,user,rank,item_update,item
0,341,1,409,410
1,341,2,455,456
2,341,3,531,532
3,341,4,359,360
4,341,5,262,263
...,...,...,...,...
100836,1021,25,318,319
100837,1021,26,116,117
100838,1021,27,69,70
100839,1021,28,28,29


In [ ]:
# Save filtered_recs to pickle
filtered_recs.to_pickle(f'{PROCESSED_FILE_DIR}/Politifact_filtered_recs_social_intersect_neighbour_0.6_threshold.pickle')

In [ ]:
# Define the different top-k values you want to generate
top_k_values = [5, 10, 15, 20]

# Loop through each k value
for k in top_k_values:
    # Filter for top-k recommendations (excluding user 0)
    MA_gnn_all_recs_top_k = filtered_recs[(filtered_recs['rank'] <= k) & (filtered_recs['user'] != 0)]
    
    # Select only the required columns: user, rank, item
    final_all_recs = MA_gnn_all_recs_top_k.iloc[:, [0, 1, 3]]
    
    # Display the final DataFrame (with only 3 columns)
    print(f"\n{'='*60}")
    print(f"TOP-{k} RECOMMENDATIONS (user, rank, item)")
    print(f"{'='*60}")
    display(final_all_recs)
    
    # Save to CSV with k in the filename
    filename = f'{DATA_FILE_DIR}/Politifact_Trustworthy_Neighbor_Space_0.6_threshold_Updated_top_{k}.csv'
    final_all_recs.to_csv(filename, index=False)
    
    print(f"\n✓ Saved {len(final_all_recs)} recommendations to {filename}")
    print(f"  - Number of users: {final_all_recs['user'].nunique()}")


TOP-5 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,341,1,410
1,341,2,456
2,341,3,532
3,341,4,360
4,341,5,263
...,...,...,...
100812,1021,1,489
100813,1021,2,189
100814,1021,3,124
100815,1021,4,522



✓ Saved 5140 recommendations to /home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/data/Politifact_Trustworthy_Neighbor_Space_0.6_threshold_Updated_top_5.csv
  - Number of users: 1028

TOP-10 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,341,1,410
1,341,2,456
2,341,3,532
3,341,4,360
4,341,5,263
...,...,...,...
100817,1021,6,272
100818,1021,7,440
100819,1021,8,532
100820,1021,9,348



✓ Saved 10280 recommendations to /home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/data/Politifact_Trustworthy_Neighbor_Space_0.6_threshold_Updated_top_10.csv
  - Number of users: 1028

TOP-15 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,341,1,410
1,341,2,456
2,341,3,532
3,341,4,360
4,341,5,263
...,...,...,...
100822,1021,11,189
100823,1021,12,394
100824,1021,13,307
100825,1021,14,356



✓ Saved 15420 recommendations to /home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/data/Politifact_Trustworthy_Neighbor_Space_0.6_threshold_Updated_top_15.csv
  - Number of users: 1028

TOP-20 RECOMMENDATIONS (user, rank, item)


,user,rank,item
0,341,1,410
1,341,2,456
2,341,3,532
3,341,4,360
4,341,5,263
...,...,...,...
100827,1021,16,440
100828,1021,17,456
100829,1021,18,173
100830,1021,19,532



✓ Saved 20560 recommendations to /home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/data/Politifact_Trustworthy_Neighbor_Space_0.6_threshold_Updated_top_20.csv
  - Number of users: 1028


## Evaluation

In [ ]:
test=pd.read_csv(f"{DATA_FILE_DIR}/politifact_test.csv")

test=test.iloc[:,[1,2,3]]
test.drop(index=1028,inplace=True)
test

,user,item,ratings
0,1,199,0.5
1,2,34,0.5
2,3,51,2.0
3,4,187,0.5
4,5,261,1.0
...,...,...,...
1023,1024,380,1.0
1024,1025,356,0.5
1025,1026,272,1.5
1026,1027,489,0.5


In [ ]:
top_k_values = [5, 10, 15, 20]

# Initialize RecListAnalysis
rla = topn.RecListAnalysis()
rla.add_metric(topn.recip_rank)

# Store results
mrr_results = {}

for k in top_k_values:
    # Load the corresponding CSV
    final_all_recs = pd.read_csv(f"{DATA_FILE_DIR}/Politifact_Trustworthy_Neighbor_Space_0.6_threshold_Updated_top_{k}.csv")
    
    # Compute MRR
    results = rla.compute(final_all_recs, test)
    mean_results = results.mean()
    
    # Check what's in the results
    print(f"\nTop-{k} Results:")
    print(mean_results)
    
    mrr_results[k] = mean_results['recip_rank']

# Create summary DataFrame
mrr_summary = pd.DataFrame(list(mrr_results.items()), columns=['Top-K', 'MRR'])
display(mrr_summary)

mrr_summary_path = os.path.join(RESULTS_DIR, 'trust_neighbour_mrr_summary_0.6_threshold.csv')
mrr_summary.to_csv(mrr_summary_path, index=False)

print(f"\n✓ MRR summary saved to {mrr_summary_path}")


Top-5 Results:
nrecs         5.000000
recip_rank    0.047568
dtype: float64

Top-10 Results:
nrecs         10.00000
recip_rank     0.05726
dtype: float64

Top-15 Results:
nrecs         15.000000
recip_rank     0.064972
dtype: float64

Top-20 Results:
nrecs         20.000000
recip_rank     0.067889
dtype: float64


,Top-K,MRR
0,5,0.047568
1,10,0.057260
2,15,0.064972
3,20,0.067889


In [ ]:
# Load the key (only once)
key = pd.read_csv("/home/shoaib/recommender-system/royal/journal-revision/src-v3/Politifact/New/data/politifact_Shu_fake_news_keyforSOCIALMF.csv")
key_news_id = key[["news_id_new","news_label"]].drop_duplicates(subset=["news_id_new"], keep="last").reset_index(drop=True)
key_news_id["fake_real_labels"] = key_news_id["news_label"].apply(lambda x: 0 if x == "fake" else 1)

# Define the MMC function
def mmc(dataframe, top_n):
    """
    Calculates the Misinformation Count (MC) divided by the top-N recommendations for each user.

    Parameters:
    dataframe (DataFrame): DataFrame with columns 'user_id', 'item_id', 'rank', 'news_label'.
    top_n (int): Number of top recommendations to consider.

    Returns:
    float: Mean MC across all users.
    """
    dataframe = dataframe[dataframe['rank'] <= top_n]
    sorted_df = dataframe.sort_values(by=['user', 'rank'])
    misinformation_df = sorted_df[sorted_df['news_label'] == 'fake']
    user_mc = misinformation_df.groupby('user').size().reset_index(name='M_item')
    user_mc['MC'] = user_mc['M_item'] / top_n
    return user_mc['MC'].mean()

# Compute MC for all top-k values
top_k_values = [5, 10, 15, 20]
mc_results = {}

for k in top_k_values:
    # Load the corresponding CSV
    final_all_recs = pd.read_csv(f"{DATA_FILE_DIR}/Politifact_Trustworthy_Neighbor_Space_0.6_threshold_Updated_top_{k}.csv")
    
    # Merge with news labels
    final_recs_mc = pd.merge(final_all_recs, key_news_id, left_on="item", right_on="news_id_new", how='inner')
    
    # Compute MC for this top-k
    mc_score = mmc(final_recs_mc, k)
    mc_results[k] = mc_score
    
    print(f"Top-{k} MC: {mc_score:.4f}")

# Create summary DataFrame
mc_summary = pd.DataFrame(list(mc_results.items()), columns=['Top-K', 'MC'])
display(mc_summary)

# Save the MC summary DataFrame to a CSV file
mc_summary_path = os.path.join(RESULTS_DIR, 'trust_neighbour_mc_summary_0.6_threshold.csv')
mc_summary.to_csv(mc_summary_path, index=False)

print(f"\n✓ MC summary saved to {mc_summary_path}")

Top-5 MC: 0.2000
Top-10 MC: 0.1972
Top-15 MC: 0.2629
Top-20 MC: 0.2986


,Top-K,MC
0,5,0.200000
1,10,0.197179
2,15,0.262905
3,20,0.298589
